# Notebook 01 — Full Keypoint Extraction

Runs the **MediaPipe Task API** (`HandLandmarker` + `PoseLandmarker` — NOT the legacy MediaPipe Holistic pipeline) on all 3,652 INCLUDE videos and saves one `.npy` file per video with shape `(T, 53, 3)`.

Joint layout (same as smoke test):
- **0–20** : left hand (21 joints)
- **21–41** : right hand (21 joints)
- **42–52** : upper body / pose (11 joints: nose, ears, shoulders, elbows, wrists, hips)

In [1]:
%pip install mediapipe opencv-python numpy pandas tqdm pyarrow --quiet

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# ── Paths ───────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("/Users/yamini/Desktop/projects/ISL PROJECT")
VIDEO_DIR    = PROJECT_ROOT / "include_videos"
MODEL_DIR    = PROJECT_ROOT          # hand_landmarker.task lives here
DATA_DIR     = PROJECT_ROOT / "data"
RAW_KP_DIR   = DATA_DIR / "raw_keypoints"
RAW_KP_DIR.mkdir(parents=True, exist_ok=True)

HAND_MODEL = str(MODEL_DIR / "hand_landmarker.task")
POSE_MODEL = str(MODEL_DIR / "pose_landmarker_full.task")

# Upper-body pose landmark indices (MediaPipe numbering → 11 joints)
# [nose, l_ear, r_ear, l_shoulder, r_shoulder, l_elbow, r_elbow,
#  l_wrist, r_wrist, l_hip, r_hip]
POSE_INDICES = [0, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24]

MAX_GAP = 5   # consecutive missing frames to interpolate (shorter = safer)

print(f"Video dir : {VIDEO_DIR}")
print(f"Output dir: {RAW_KP_DIR}")


Video dir : /Users/yamini/Desktop/projects/ISL PROJECT/include_videos
Output dir: /Users/yamini/Desktop/projects/ISL PROJECT/data/raw_keypoints


In [3]:
# ── Build MediaPipe detectors ────────────────────────────────────────────────
hand_detector = vision.HandLandmarker.create_from_options(
    vision.HandLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=HAND_MODEL),
        num_hands=2,
        min_hand_detection_confidence=0.3,
        min_hand_presence_confidence=0.3,
        min_tracking_confidence=0.3,
    )
)

pose_detector = vision.PoseLandmarker.create_from_options(
    vision.PoseLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=POSE_MODEL),
        num_poses=1,
        min_pose_detection_confidence=0.3,
        min_pose_presence_confidence=0.3,
        min_tracking_confidence=0.3,
    )
)
print("MediaPipe detectors ready.")

I0000 00:00:1787428221.338297 37589956 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 13, prefix = pthread-default
I0000 00:00:1787428221.437099 37589956 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1787428221.450145 37589958 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787428221.468330 37589966 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1787428221.486562 37589974 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro


MediaPipe detectors ready.


W0000 00:00:1787428221.558592 37589979 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787428221.580854 37589978 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [4]:
def fix_missing_hand(hand_kps, max_gap=MAX_GAP):
    """
    hand_kps : ndarray (T, 21, 3)

    A frame is "missing" when every joint is exactly (0, 0, 0).
    Short gaps (length <= max_gap) that have valid frames on BOTH sides
    are filled via linear interpolation.
    Longer gaps are left as zeros — they represent a genuinely inactive
    hand or an irrecoverable detection failure.

    Returns a new (T, 21, 3) array.
    """
    T = hand_kps.shape[0]
    missing = np.all(hand_kps == 0.0, axis=(1, 2))   # (T,) bool

    if not missing.any():
        return hand_kps

    result = hand_kps.copy()
    i = 0
    while i < T:
        if not missing[i]:
            i += 1
            continue
        # find the end of this contiguous gap
        j = i
        while j < T and missing[j]:
            j += 1
        gap_len   = j - i
        has_before = (i > 0) and (not missing[i - 1])
        has_after  = (j < T) and (not missing[j])
        if gap_len <= max_gap and has_before and has_after:
            before = result[i - 1]          # (21, 3) — last valid frame
            after  = result[j]              # (21, 3) — next valid frame
            for k in range(gap_len):
                alpha = (k + 1) / (gap_len + 1)
                result[i + k] = (1.0 - alpha) * before + alpha * after
        i = j   # jump past the gap
    return result

In [5]:
def extract_keypoints(video_path):
    """
    Extracts 53 keypoints per frame from a video file.

    Returns
    -------
    kps        : ndarray (T, 53, 3)  float32
    stats_raw  : dict  —  missing frame counts BEFORE interpolation
    stats_fixed: dict  —  missing frame counts AFTER  interpolation
    """
    cap = cv2.VideoCapture(str(video_path))
    raw_frames = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

        # ── Hands ──────────────────────────────────────────────────────────
        hand_result = hand_detector.detect(mp_image)
        left_hand   = np.zeros((21, 3), dtype=np.float32)
        right_hand  = np.zeros((21, 3), dtype=np.float32)
        for idx, handedness_list in enumerate(hand_result.handedness):
            label = handedness_list[0].category_name   # "Left" or "Right"
            lms   = np.array(
                [[lm.x, lm.y, lm.z] for lm in hand_result.hand_landmarks[idx]],
                dtype=np.float32
            )
            if label == "Left":
                left_hand = lms
            else:
                right_hand = lms

        # ── Pose (upper body) ───────────────────────────────────────────────
        pose_result = pose_detector.detect(mp_image)
        if pose_result.pose_landmarks:
            lms  = pose_result.pose_landmarks[0]
            body = np.array(
                [[lms[i].x, lms[i].y, lms[i].z] for i in POSE_INDICES],
                dtype=np.float32
            )
        else:
            body = np.zeros((11, 3), dtype=np.float32)

        # 53 joints = left(21) + right(21) + body(11)
        raw_frames.append(np.concatenate([left_hand, right_hand, body], axis=0))

    cap.release()

    if len(raw_frames) == 0:
        return None, {}, {}

    kps = np.stack(raw_frames, axis=0).astype(np.float32)   # (T, 53, 3)

    # ── Stats before fix ────────────────────────────────────────────────────
    miss_left_raw  = int(np.sum(np.all(kps[:, :21,  :] == 0, axis=(1, 2))))
    miss_right_raw = int(np.sum(np.all(kps[:, 21:42, :] == 0, axis=(1, 2))))
    stats_raw = {"missing_left": miss_left_raw, "missing_right": miss_right_raw}

    # ── Apply missing-hand fix ──────────────────────────────────────────────
    kps[:, :21,  :] = fix_missing_hand(kps[:, :21,  :])    # left hand
    kps[:, 21:42, :] = fix_missing_hand(kps[:, 21:42, :])  # right hand

    # ── Stats after fix ─────────────────────────────────────────────────────
    miss_left_fix  = int(np.sum(np.all(kps[:, :21,  :] == 0, axis=(1, 2))))
    miss_right_fix = int(np.sum(np.all(kps[:, 21:42, :] == 0, axis=(1, 2))))
    stats_fixed = {"missing_left": miss_left_fix, "missing_right": miss_right_fix}

    return kps, stats_raw, stats_fixed

In [6]:
# ── Scan all videos ──────────────────────────────────────────────────────────
# Recursive (rglob) rather than a fixed 2-level walk: some sign folders in the
# official INCLUDE archive nest extra takes one level deeper in an "Extra"/
# "extra" subfolder (27 videos across Adjectives/Animals/Places, all present
# in the official train_test_paths lists). A fixed cat_dir -> sign_dir -> glob
# walk silently misses these; rglob + taking the first path component under
# the category as the sign name handles both layouts uniformly.
all_videos = []
for cat_dir in sorted(VIDEO_DIR.iterdir()):
    if not cat_dir.is_dir():
        continue
    for video_file in sorted(cat_dir.rglob("*.MOV")):
        rel_to_cat = video_file.relative_to(cat_dir)
        sign_name  = rel_to_cat.parts[0]   # sign folder is always the first component
        all_videos.append({
            "category"  : cat_dir.name,
            "sign"      : sign_name,
            "video_id"  : video_file.stem,          # e.g. "MVI_2978"
            "video_path": str(video_file),
            # relative path matching the official train_test_paths format
            "rel_path"  : f"{cat_dir.name}/{rel_to_cat.as_posix()}",
        })

print(f"Found {len(all_videos)} videos across "
      f"{len(set(v['category'] for v in all_videos))} categories and "
      f"{len(set(v['sign'] for v in all_videos))} sign classes.")

Found 4284 videos across 15 categories and 262 sign classes.


In [7]:
# ── Main extraction loop (resumable) ────────────────────────────────────────
# Each video is saved as:
#   data/raw_keypoints/<category>__<sign>__<video_id>.npy   shape (T, 53, 3)

metadata = []
errors   = []

for v in tqdm(all_videos, desc="Extracting keypoints"):
    safe_sign = v["sign"].replace("/", "-")   # avoid sub-path issues
    out_name  = f"{v['category']}__{safe_sign}__{v['video_id']}.npy"
    out_path  = RAW_KP_DIR / out_name

    # Resumable: skip already-done files
    if out_path.exists():
        existing = np.load(str(out_path), mmap_mode="r")
        row = {**v, "n_frames": existing.shape[0], "out_name": out_name,
               "status": "skipped"}
        metadata.append(row)
        continue

    try:
        kps, stats_raw, stats_fixed = extract_keypoints(v["video_path"])
        if kps is None:
            errors.append({"video": v["video_path"], "error": "No frames decoded"})
            continue
        np.save(str(out_path), kps)
        row = {**v,
               "n_frames"          : kps.shape[0],
               "out_name"          : out_name,
               "status"            : "ok",
               "miss_left_raw"     : stats_raw["missing_left"],
               "miss_right_raw"    : stats_raw["missing_right"],
               "miss_left_fixed"   : stats_fixed["missing_left"],
               "miss_right_fixed"  : stats_fixed["missing_right"]}
        metadata.append(row)
    except Exception as e:
        errors.append({"video": v["video_path"], "error": str(e)})
        print(f"ERROR: {v['video_path']}\n  {e}")

print(f"\nDone.")
print(f"  Processed : {sum(1 for m in metadata if m.get('status') == 'ok')}")
print(f"  Skipped   : {sum(1 for m in metadata if m.get('status') == 'skipped')}")
print(f"  Errors    : {len(errors)}")

Extracting keypoints:   0%|          | 0/4284 [00:00<?, ?it/s]

W0000 00:00:1787428221.917637 37589966 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.



Done.
  Processed : 27
  Skipped   : 4257
  Errors    : 0


In [8]:
# ── Save metadata CSV ────────────────────────────────────────────────────────
meta_df = pd.DataFrame(metadata)
meta_df.to_csv(DATA_DIR / "keypoints_metadata.csv", index=False)

print("=== Frame count statistics ===")
print(meta_df["n_frames"].describe().round(1))

if "miss_left_raw" in meta_df.columns:
    ok = meta_df[meta_df.status == "ok"]
    print("\n=== Missing hand frames BEFORE interpolation (mean per video) ===")
    print(f"  Left hand  : {ok['miss_left_raw'].mean():.1f} frames")
    print(f"  Right hand : {ok['miss_right_raw'].mean():.1f} frames")
    print("\n=== Missing hand frames AFTER interpolation ===")
    print(f"  Left hand  : {ok['miss_left_fixed'].mean():.1f} frames")
    print(f"  Right hand : {ok['miss_right_fixed'].mean():.1f} frames")

print(f"\nMetadata saved to: {DATA_DIR / 'keypoints_metadata.csv'}")
meta_df.head()

=== Frame count statistics ===
count    4284.0
mean       64.3
std        15.0
min        33.0
25%        54.0
50%        62.0
75%        71.2
max       154.0
Name: n_frames, dtype: float64

=== Missing hand frames BEFORE interpolation (mean per video) ===
  Left hand  : 18.8 frames
  Right hand : 18.0 frames

=== Missing hand frames AFTER interpolation ===
  Left hand  : 11.4 frames
  Right hand : 9.6 frames

Metadata saved to: /Users/yamini/Desktop/projects/ISL PROJECT/data/keypoints_metadata.csv


,category,sign,video_id,video_path,rel_path,n_frames,out_name,status,miss_left_raw,miss_right_raw,miss_left_fixed,miss_right_fixed
0,Adjectives,1. loud,MVI_5177,/Users/yamini/Desktop/projects/ISL PROJECT/inc...,Adjectives/1. loud/MVI_5177.MOV,56,Adjectives__1. loud__MVI_5177.npy,skipped,NaN,NaN,NaN,NaN
1,Adjectives,1. loud,MVI_5178,/Users/yamini/Desktop/projects/ISL PROJECT/inc...,Adjectives/1. loud/MVI_5178.MOV,64,Adjectives__1. loud__MVI_5178.npy,skipped,NaN,NaN,NaN,NaN
2,Adjectives,1. loud,MVI_5179,/Users/yamini/Desktop/projects/ISL PROJECT/inc...,Adjectives/1. loud/MVI_5179.MOV,66,Adjectives__1. loud__MVI_5179.npy,skipped,NaN,NaN,NaN,NaN
3,Adjectives,1. loud,MVI_5257,/Users/yamini/Desktop/projects/ISL PROJECT/inc...,Adjectives/1. loud/MVI_5257.MOV,52,Adjectives__1. loud__MVI_5257.npy,skipped,NaN,NaN,NaN,NaN
4,Adjectives,1. loud,MVI_5258,/Users/yamini/Desktop/projects/ISL PROJECT/inc...,Adjectives/1. loud/MVI_5258.MOV,76,Adjectives__1. loud__MVI_5258.npy,skipped,NaN,NaN,NaN,NaN


In [9]:
# ── Quick sanity check — load one file and verify shape ─────────────────────
import random
# Accept both 'ok' (freshly extracted) and 'skipped' (already existed on disk)
sample = random.choice([m for m in metadata if m.get('status') in ('ok', 'skipped')])
kps = np.load(str(RAW_KP_DIR / sample['out_name']))
print(f"Sample: {sample['category']} / {sample['sign']} / {sample['video_id']}")
print(f"  Shape   : {kps.shape}   (frames, joints, coords)")
print(f"  dtype   : {kps.dtype}")
print(f"  min/max : {kps.min():.4f} / {kps.max():.4f}")
assert kps.shape[1] == 53 and kps.shape[2] == 3, "Unexpected shape!"
print("Shape check PASSED.")


Sample: Home / 31. Kitchen / MVI_9024
  Shape   : (75, 53, 3)   (frames, joints, coords)
  dtype   : float32
  min/max : -0.5166 / 0.8556
Shape check PASSED.
